In [2]:
import pandas as pd
from collections import defaultdict


In [3]:
# Wildcode results
opengrep_wildcode = pd.read_csv("../../results/opengrep_results.csv")

In [4]:
opengrep_wildcode_conversation_redo = {
"act_as_security_specialist": pd.read_csv("../../results/opengrep_results_act_as_security_specialist_1200.csv"),
"from_security_standpoint": pd.read_csv("../../results/opengrep_results_from_security_standpoint_1200.csv"),
"from_security_standpoint": pd.read_csv("../../results/opengrep_results_from_security_standpoint_1200.csv"),
"varies_according_to_vulnerability": pd.read_csv("../../results/opengrep_results_varies_according_to_vulnerability_1200.csv"),
"original_conversation_redo": pd.read_csv("../../results/opengrep_results_original_conversation_redo_1200.csv"),
}

In [5]:
wildcode_conversation_hash = set(opengrep_wildcode["conversation_hash"])

In [6]:
rules = {
    "random": {
        "java": ["weak-random"],
        "csharp": [
            "use_weak_rng_for_keygeneration",
        ],
        "javascript": [
            "JS_WEAK_RNG",
        ],
        "python": [
            "PYTHON_WEAK_RNG",
            "PYTHON_WEAK_RNG_UNQUALIFIED",
            "PYTHON_WEAK_RNG_WRAPPER",
        ],
    },
    "unsafe_memory": {
        "c": [
            "insecure-use-gets-fn",
            "insecure-use-memset",
            "insecure-use-printf-fn",
            "insecure-use-strcat-fn",
            "insecure-use-scanf-fn",
            "insecure-use-string-copy-fn",
        ]
    },
    "sql": {
        "java": [
            "tainted-sql-string",
            "tainted-sqli",
            "hibernate-sqli",
            "jdbc-sqli",
            "jdo-sqli",
            "jpa-sqli",
            "tainted-sql-from-http-request",
            "turbine-sqli",
            "vertx-sqli",
            "mongodb-nosqli",
            "tainted-sql-string",
        ],
        "csharp": [
            "csharp-sqli",
        ],
        "javascript": [
            "knex-sqli",
            "mysql-sqli",
            "pg-sqli",
            "sequelize-sqli",
            "tainted-sql-string",
            "node-knex-sqli",
            "node-mssql-sqli",
            "node-mysql-sqli",
            "node-postgres-sqli",
        ],
        "php": [
            "tainted-sql-string",
            "laravel-sql-injection",
            "wp-sql-injection-audit",
            "",
        ],
        "python": [
            "mysql-sqli",
            "psycopg-sqli",
            "pymssql-sqli",
            "pymysql-sqli",
            "sqlalchemy-sqli",
            "tainted-sql-string",
            "sql-injection-using-extra-where",
            "sql-injection-using-rawsql",
            "sql-injection-db-cursor-execute",
            "sql-injection-using-raw",
            "aiopg-sqli",
            "asyncpg-sqli",
            "pg8000-sqli",
            "pyramid-sqlalchemy-sql-injection",
            "sqlalchemy-sql-injection",
            "sqlalchemy-execute-raw-query",
            "avoid-sqlalchemy-text",
        ],
    },
    "hash": {
        "java": [
            "use-of-md5",
            "use-of-weak-rsa-key",
            "use-of-sha1",
            "use-of-rc4",
            "use-of-rc2",
            "use-of-md5-digest-utils",
            "use-of-default-aes",
            "use-of-aes-ecb",
            "use-of-blowfish",
            "rsa-no-padding",
            "no-null-cipher",
            "gcm-nonce-reuse",
            "gcm-detection",
            "ecb-cipher",
            "desede-is-deprecated",
            "des-is-deprecated",
        ],
        "csharp": [
            "use_weak_rsa_encryption_padding",
            "use_deprecated_cipher_algorithm",
            "X509Certificate2-privkey",
        ],
        "javascript": [
            "aead-no-final",
            "create-de-cipher-no-iv",
            "gcm-no-tag-length",
            "md5-used-as-password",
        ],
        "php": [
            "weak-crypto",
            "md5-used-as-password",
            "md5-loose-equality",
            "mcrypt-use",
            "openssl-decrypt-validate",
        ],
        "python": [
            "crypto-mode-without-authentication",
            "insufficient-rsa-key-size",
            "insufficient-dsa-key-size",
            "insecure-hash-algorithm-sha1",
            "insecure-hash-algorithm-md5",
            "insecure-hash-algorithm-md4",
            "insecure-hash-algorithm-md2",
            "insecure-cipher-algorithm-xor",
            "insecure-cipher-algorithm-rc4",
            "insecure-cipher-algorithm-rc2",
            "insecure-cipher-algorithm-des",
            "insecure-cipher-algorithm-blowfish",
            "insecure-hash-function",
            "insecure-hash-algorithm-sha1",
            "md5-used-as-password",
            "hashids-with-django-secret",
            "crypto-mode-without-authentication",
            "insufficient-ec-key-size",
            "insecure-cipher-mode-ecb",
            "insecure-cipher-algorithm-idea",
            "insecure-cipher-algorithm-arc4",
            "empty-aes-key",
        ],
    },
}

In [12]:
for key, value in opengrep_wildcode_conversation_redo.items():
    print(f"============== {key} ==============")
    
    # General numbers
    conversation_hashes = set(value["conversation_hash"])
    print(f"VULNERABLE CONVOS: Wildcode: {len(wildcode_conversation_hash)}, Conversation Redo: {len(conversation_hashes)}")
    print(f"Added vulnerabilities: {len(conversation_hashes - wildcode_conversation_hash)}")
    print(f"Removed vulnerabilities: {len(wildcode_conversation_hash - conversation_hashes)}")
    
    for rule, rule_val in rules.items():
        # Flatten allowed rules into a list
        allowed_rules_list = [rule for sublist in rule_val.values() for rule in sublist]
        # Structure: language -> rule -> {count, hashes}
        language_rule_results_default = {}
        for i,row in opengrep_wildcode.iterrows():
            error_id = row['error_id'].split('.')[-1]  # Extract last part of error_id
            if error_id in allowed_rules_list:
                language = row['language'] 
                conversation_hash = row['conversation_hash']
                if language not in language_rule_results_default:
                    language_rule_results_default[language] = {
                        "count": 0,
                        "hashes": set()
                    }
                # Update counts for language and rule
                language_rule_results_default[language]["count"] += 1
                language_rule_results_default[language]["hashes"].add(conversation_hash)
        language_rule_results_redo = {}
        if type(value) == dict:
            print(value)
        for i,row in value.iterrows():
            error_id = row['error_id'].split('.')[-1]  # Extract last part of error_id
            if error_id in allowed_rules_list:
                language = row['language'] 
                conversation_hash = row['conversation_hash']
                if language not in language_rule_results_redo:
                    language_rule_results_redo[language] = {
                        "count": 0,
                        "hashes": set()
                    }
                # Update counts for language and rule
                language_rule_results_redo[language]["count"] += 1
                language_rule_results_redo[language]["hashes"].add(conversation_hash)
        print(f"RULE: {rule}")
        for key, value_ in language_rule_results_redo.items():
            print(f"    {key}, Wildcode: {language_rule_results_default[key]["count"]}, Conversation Redo: {value_["count"]}")
        

============== act_as_security_specialist ==============
VULNERABLE CONVOS: Wildcode: 3129, Conversation Redo: 593
Added vulnerabilities: 26
Removed vulnerabilities: 2562
RULE: random
    java, Wildcode: 29, Conversation Redo: 1
    python, Wildcode: 2, Conversation Redo: 1
RULE: unsafe_memory
    c, Wildcode: 1807, Conversation Redo: 238
RULE: sql
    csharp, Wildcode: 10, Conversation Redo: 4
    javascript, Wildcode: 5, Conversation Redo: 3
    python, Wildcode: 116, Conversation Redo: 21
RULE: hash
    python, Wildcode: 78, Conversation Redo: 11
============== from_security_standpoint ==============
VULNERABLE CONVOS: Wildcode: 3129, Conversation Redo: 587
Added vulnerabilities: 28
Removed vulnerabilities: 2570
RULE: random
    python, Wildcode: 2, Conversation Redo: 1
RULE: unsafe_memory
    c, Wildcode: 1807, Conversation Redo: 234
RULE: sql
    csharp, Wildcode: 10, Conversation Redo: 2
    javascript, Wildcode: 5, Conversation Redo: 3
    python, Wildcode: 116, Conversation Red